In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(8)
size = (512, 512)
n = np.prod(size)


In [ ]:
a, b = -1.0, 1.0
u = rng.random(n)
uniform_noise = a + (b - a) * u
cv_uniform = np.empty(n, dtype=np.float32)
cv2.randu(cv_uniform, a, b)


In [ ]:
mu, sigma = 0.0, 0.2
u1 = np.maximum(rng.random(n), np.finfo(float).tiny)
u2 = rng.random(n)
gaussian_noise = mu + sigma * np.sqrt(-2 * np.log(u1)) * np.cos(2 * np.pi * u2)
cv_gaussian = np.empty(n, dtype=np.float32)
cv2.randn(cv_gaussian, mu, sigma)


In [ ]:
k, rate = 3, 2.0
erlang_noise = -np.log(np.maximum(rng.random((k, n)), np.finfo(float).tiny)).sum(axis=0) / rate


In [ ]:
rate_exp = 2.0
exponential_noise = -np.log1p(-rng.random(n)) / rate_exp


In [ ]:
rayleigh_sigma = 1.0
rayleigh_noise = rayleigh_sigma * np.sqrt(-2 * np.log1p(-rng.random(n)))


In [ ]:
samples = [("Uniform", uniform_noise, cv_uniform), ("Gaussian", gaussian_noise, cv_gaussian), ("Erlang", erlang_noise, None), ("Exponential", exponential_noise, None), ("Rayleigh", rayleigh_noise, None)]
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for ax, (name, generated, builtin) in zip(axes.flat, samples):
    ax.hist(generated, bins=80, density=True, alpha=0.65, label="Equation")
    if builtin is not None:
        ax.hist(builtin, bins=80, density=True, histtype="step", linewidth=1.5, label="OpenCV built-in")
    ax.set_title(name + (" (equation vs OpenCV)" if builtin is not None else " (equation)"))
    ax.set_xlabel("Noise value"); ax.set_ylabel("Density"); ax.legend()
axes.flat[-1].axis("off")
plt.tight_layout(); plt.show()

print("Equation and built-in means / standard deviations:")
for name, generated, builtin in samples[:2]:
    print(f"{name}: equation ({generated.mean():.4f}, {generated.std():.4f}); built-in ({builtin.mean():.4f}, {builtin.std():.4f})")